# Analyzing the shell

In [1]:
import sys

import micromagneticmodel as mm
import numpy as np
import pyvista as pv

import tommos_adapter as ta

## Generate mesh

In [2]:
!tommos mesh --geom box --extent 20,20,20 --h 2 --out-name cube

[ok] Wrote data: cube.npz (nodes=3284, tets=15109)
[ok] Wrote visualization: cube.vtu (cell_data: mat_id)


## Define magnetization (0, 1, 0)

In [3]:
mesh_input = pv.read("cube.vtu")
mesh_input.field_data["mesh_unit_length"] = 1e-9
mesh_input.point_data["m"] = np.tile((0.0, 1.0, 0.0), (mesh_input.n_points, 1))
mesh_input.cell_data["Ms"] = np.full(mesh_input.n_cells, 1.76 / mm.consts.mu0)  # spontaneous polarization = 1.76 Tesla
mesh_input

UnstructuredGrid (0x7f91d161bdc0)
  N Cells:    15109
  N Points:   3284
  X Bounds:   -1.000e+01, 1.000e+01
  Y Bounds:   -1.000e+01, 1.000e+01
  Z Bounds:   -1.000e+01, 1.000e+01
  N Arrays:   4

In [4]:
mesh_input.glyph(orient="m", scale=None, tolerance=0.05).plot()

2026-11-10 13:37:59.376 (   5.484s) [    7F9232919740]vtkXOpenGLRenderWindow.:1452  WARN| bad X server connection. DISPLAY=


JS Error => no webgl context
JS Error => TypeError: `target` argument of Proxy must be an object, got null
JS Error => no webgl context
JS Error => TypeError: `target` argument of Proxy must be an object, got null
JS Error => no webgl context
JS Error => TypeError: `target` argument of Proxy must be an object, got null
JS Error => no webgl context
JS Error => TypeError: `target` argument of Proxy must be an object, got null


## Initialize System

In [5]:
system = mm.System(name="analyze_shell")
system.m = mesh_input.copy()

## Define energy terms

In [6]:
A = 1e-12  # exchange stiffness constant in J/m
K = 4e5  # first uniaxial anisotropy constant in J/m^3
u = (0, 0, 1)  # anisotropy unit axis
system.energy = mm.Exchange(A=A) + mm.UniaxialAnisotropy(K=K, u=u) + mm.Demag()
system.energy

Exchange(A=1e-12) + UniaxialAnisotropy(K=400000.0, u=(0, 0, 1)) + Demag()

## Drive hysteresis loop

In [7]:
hd = ta.HysteresisDriver()
hd.drive(system, h_direction=(1, 0, 0), h_start=-2 / mm.consts.mu0, h_final=1 / mm.consts.mu0, n=16)

Running Tommos (TommosRunner)[2026-11-10T13:37:59]... (16.3 s)


## Output system magnetization (with shell)

The mesh saved by `tommos` includes the airbox shell. The `mat_id` of the shell seems to be the highest integer.

In [8]:
out_with_shell = pv.read("analyze_shell/drive-0/hyst_analyze_shell/state_cfg00015_B+1.0000e+00T.vtu")
out_with_shell

UnstructuredGrid (0x7f91b2a0ef80)
  N Cells:    37027
  N Points:   6258
  X Bounds:   -1.139e+02, 1.139e+02
  Y Bounds:   -1.139e+02, 1.139e+02
  Z Bounds:   -1.139e+02, 1.139e+02
  N Arrays:   3

The glyph plots both the magnetic material and the shell.

In [9]:
out_with_shell.glyph(orient="m", scale=None, tolerance=0.0001).plot()

## Remove shell

We remove the shell by only selecting cells with the ID of the input mesh:

In [10]:
original_ids = np.unique(mesh_input["mat_id"])  # we extract the original IDs without knowing them explicitly
valid_cells = out_with_shell.cell_data["mat_id"] == original_ids
out_without_shell = out_with_shell.extract_cells(valid_cells)
out_without_shell

UnstructuredGrid (0x7f91b293fac0)
  N Cells:    15109
  N Points:   3284
  X Bounds:   -1.000e+01, 1.000e+01
  Y Bounds:   -1.000e+01, 1.000e+01
  Z Bounds:   -1.000e+01, 1.000e+01
  N Arrays:   5

In [11]:
out_without_shell.glyph(orient="m", scale=None, tolerance=0.04).plot()

One arrow for each cell:

In [12]:
out_without_shell.point_data_to_cell_data().glyph(orient="m", scale=None, tolerance=0.04).plot()

# General comparison between `mesh_input` and `out_without_shell`

In [13]:
mesh_input

UnstructuredGrid (0x7f91d161bdc0)
  N Cells:    15109
  N Points:   3284
  X Bounds:   -1.000e+01, 1.000e+01
  Y Bounds:   -1.000e+01, 1.000e+01
  Z Bounds:   -1.000e+01, 1.000e+01
  N Arrays:   4

In [14]:
out_with_shell

UnstructuredGrid (0x7f91b2a0ef80)
  N Cells:    37027
  N Points:   6258
  X Bounds:   -1.139e+02, 1.139e+02
  Y Bounds:   -1.139e+02, 1.139e+02
  Z Bounds:   -1.139e+02, 1.139e+02
  N Arrays:   3

In [15]:
out_without_shell

UnstructuredGrid (0x7f91b293fac0)
  N Cells:    15109
  N Points:   3284
  X Bounds:   -1.000e+01, 1.000e+01
  Y Bounds:   -1.000e+01, 1.000e+01
  Z Bounds:   -1.000e+01, 1.000e+01
  N Arrays:   5

## Comparison of input and output (without shell) mesh points

In [16]:
mesh_input.points

pyvista_ndarray([[-10.        , -10.        , -10.        ],
                 [ 10.        , -10.        , -10.        ],
                 [ 10.        ,  10.        , -10.        ],
                 ...,
                 [ -9.40025703,  -6.79246162, -10.        ],
                 [ -9.4982616 ,  -7.31540747, -10.        ],
                 [ -9.3909605 ,  -7.53230753,  -9.27043323]],
                shape=(3284, 3))

In [17]:
out_without_shell.points

pyvista_ndarray([[-10.  , -10.  , -10.  ],
                 [-10.  , -10.  ,  -8.75],
                 [-10.  , -10.  ,  -7.5 ],
                 ...,
                 [ 10.  ,  10.  ,   5.  ],
                 [ 10.  ,  10.  ,   7.5 ],
                 [ 10.  ,  10.  ,  10.  ]], shape=(3284, 3), dtype=float32)

The input mesh and output (without shell) mesh are shuffled:

In [18]:
mesh_input.points == out_without_shell.points

pyvista_ndarray([[ True,  True,  True],
                 [False,  True, False],
                 [False, False, False],
                 ...,
                 [False, False, False],
                 [False, False, False],
                 [False, False, False]], shape=(3284, 3))

Example:

In [19]:
mesh_input.points[104]

pyvista_ndarray([-1.7849848 ,  0.12817606, -1.31076283])

In [20]:
out_without_shell.points[104]

pyvista_ndarray([-10.        ,  -2.9561017 ,  -0.09847778], dtype=float32)

## On the contiguity of shell points

The shell points seem to be all appended at the end:

In [21]:
np.set_printoptions(threshold=sys.maxsize)
out_with_shell.cell_data["mat_id"]

pyvista_ndarray([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 

## The cells are the same:

In [22]:
mesh_input.get_cell(204)

Cell (0x7f91b101d360)
  Type:        <CellType.TETRA: 10>
  Linear:      True
  Dimension:   3
  N Points:    4
  N Faces:     4
  N Edges:     6
  X Bounds:    -4.750e+00, -3.296e+00
  Y Bounds:    5.337e+00, 7.055e+00
  Z Bounds:    3.457e-01, 1.569e+00

In [23]:
out_with_shell.get_cell(204)

Cell (0x7f91b50ab580)
  Type:        <CellType.TETRA: 10>
  Linear:      True
  Dimension:   3
  N Points:    4
  N Faces:     4
  N Edges:     6
  X Bounds:    -4.750e+00, -3.296e+00
  Y Bounds:    5.337e+00, 7.055e+00
  Z Bounds:    3.457e-01, 1.569e+00

In [24]:
out_without_shell.get_cell(204)

Cell (0x7f91b101d180)
  Type:        <CellType.TETRA: 10>
  Linear:      True
  Dimension:   3
  N Points:    4
  N Faces:     4
  N Edges:     6
  X Bounds:    -4.750e+00, -3.296e+00
  Y Bounds:    5.337e+00, 7.055e+00
  Z Bounds:    3.457e-01, 1.569e+00

## The points are not the same:

In [25]:
mesh_input.points[204]

pyvista_ndarray([ 5., 10., 10.])

In [26]:
out_with_shell.points[204]

pyvista_ndarray([-64.947914,  16.236979, -48.710938], dtype=float32)

In [27]:
out_without_shell.points[204]

pyvista_ndarray([-10. ,   7.5,  10. ], dtype=float32)

## Extract by vicinity
Basic solution to extract from the output mesh only the points closest to the input mesh:

In [28]:
mesh_output = mesh_input.copy()
for i, point in enumerate(mesh_output.points):
    mesh_output.point_data["m"][i] = out_with_shell.point_data["m"][out_with_shell.find_closest_point(point)]
mesh_output

UnstructuredGrid (0x7f91b2a0e980)
  N Cells:    15109
  N Points:   3284
  X Bounds:   -1.000e+01, 1.000e+01
  Y Bounds:   -1.000e+01, 1.000e+01
  Z Bounds:   -1.000e+01, 1.000e+01
  N Arrays:   4

In [29]:
%%timeit
for i, point in enumerate(mesh_output.points):
    mesh_output.point_data["m"][i] = out_with_shell.point_data["m"][out_with_shell.find_closest_point(point)]

168 ms ± 1.36 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


Not as fast as current solution but correct in case the output mesh does not contain the input mesh exactly.

In [30]:
(mesh_output.points == mesh_input.points).all()

np.True_

In [31]:
(mesh_output.points == system.m.points).all()

np.True_